# 1. Scanner

In [ ]:
import os 
from dotenv import load_dotenv
from newsapi import NewsApiClient 

secret_file = "../secrets/news_api_key.env"
loaded = load_dotenv(secret_file, override=True)
NEWS_API_KEY = os.getenv("NEWS_API_KEY")

if not loaded:
    raise FileNotFoundError(f"Could not load {secret_file}")

if not NEWS_API_KEY:
    raise ValueError("NEWS_API_KEY not found in environment file")

In [ ]:
import requests
from requests import Response
from datetime import datetime, timezone, timedelta
from typing import List, Dict, Any, Optional

class NewsScanner:
    def __init__(self, api_key: str) -> None:
        self.api_key = api_key
        self.base_url = "https://newsapi.org/v2/everything"
    
    def fetch_news(self, query: str, days_back = 3) -> List[Dict[str, Any]]:
        from_date = (datetime.now(timezone.utc) 
                    - timedelta(days=days_back)).strftime("%Y-%m-%d")

        params = {"q": query,
                  "from": from_date,
                  "sortBy": "publishedAt",
                  "language": "en",
                  "apiKey": self.api_key,
                  "pageSize": 10}
        
        response : Response = requests.get(self.base_url, params=params)
        data : Dict[str, Any] = response.json()

        if data.get("status") != "ok":
            raise Exception(f"NewsAPI error: {data}")
        
        articles: List[Dict[str, Any]] = data.get("articles", [])
        return self.normalize(articles)

    def normalize(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Optional[str]]]:
        normalized : List[Dict[str, Optional[str]]] = []

        for article in articles:
            normalized.append({"title": article.get("title"),
                               "description": article.get("description"),
                               "source": article.get("source", {}).get("name"),
                               "published_at": article.get("publishedAt"),
                               "url": article.get("url")
            })
        
        return normalized

In [ ]:
import textwrap

def print_event(event: Dict[str, Optional[str]]) -> None:
    print("\n" + "-" * 100)
    print("TITLE:")
    print(textwrap.fill(event.get("title", ""), 80))

    print("\nSOURCE:", event.get("source"))
    print("DATE:", event.get("published_at"))

    print("\nDESCRIPTION:")
    print(textwrap.fill(event.get("description", ""), 80))

    print("\nURL:", event.get("url"))

In [ ]:
query = "EU AI Act"
scanner : NewsScanner = NewsScanner(NEWS_API_KEY)
events : List[Dict[str, Optional[str]]] = scanner.fetch_news(query)

for event in events[:5]:
    print_event(event)


----------------------------------------------------------------------------------------------------
TITLE:
EU rules reining in Big Tech will now target cloud services and AI, regulators
say

SOURCE: The Times of India
DATE: 2026-04-28T17:39:54Z

DESCRIPTION:
The European Union is expanding its Digital Markets Act to cover cloud and
artificial intelligence services. This move aims to ensure fairer competition in
these growing digital sectors. Regulators are examining if major companies like
Amazon and Microsoft sh…

URL: https://economictimes.indiatimes.com/tech/technology/eu-rules-reining-in-big-tech-will-now-target-cloud-services-and-ai-regulators-say/articleshow/130587317.cms

----------------------------------------------------------------------------------------------------
TITLE:
EU rules reining in Big Tech will now target cloud services and AI, regulators
say

SOURCE: CNA
DATE: 2026-04-28T17:06:47Z

DESCRIPTION:
BRUSSELS, April 28 : The European Union plans to turn the focus o

# 4. Generator

### 4.1 LinkedIn Posts - Preprocessing

In [20]:
import pandas as pd

metrics_df = pd.read_excel("../data/KickstartAI LinkedIn post year to date.xlsx", 
                            sheet_name="Metrics",
                            skiprows=1)
metrics_df.head(3)

,Date,Impressions (organic),Impressions (sponsored),Impressions (total),Unique impressions (organic),Clicks (organic),Clicks (sponsored),Clicks (total),Reactions (organic),Reactions (sponsored),Reactions (total),Comments (organic),Comments (sponsored),Comments (total),Reposts (organic),Reposts (sponsored),Reposts (total),Engagement rate (organic),Engagement rate (sponsored),Engagement rate (total)
0,04/21/2025,232,6756,6988,131,6,11,17,1,1,2,1,0,1,0,0,0,0.034483,0.004144,0.005152
1,04/22/2025,1197,7165,8362,580,65,6,71,12,0,12,0,0,0,0,0,0,0.064327,0.002373,0.011241
2,04/23/2025,507,4,511,226,36,0,36,5,0,5,0,0,0,2,0,2,0.084813,0.000000,0.084149


In [24]:
posts_df = pd.read_excel("../data/KickstartAI LinkedIn post year to date.xlsx", 
                         sheet_name="All posts",
                         skiprows=1)
posts_df.head(3)

,Post title,Post link,Post type,Campaign name,Posted by,Created date,Campaign start date,Campaign end date,Audience,Impressions,Views,Offsite Views,Clicks,Click through rate (CTR),Likes,Comments,Reposts,Follows,Engagement rate,Content Type
0,AI is high on every agenda. But for many organ...,https://www.linkedin.com/feed/update/urn:li:ac...,Organic,NaN,Ioanna Lykiardopoulou,04/16/2026,NaN,NaN,All followers,0,NaN,NaN,0,0.0,21,0,0,NaN,0.000000,NaN
1,AI is high on every agenda. But for many organ...,https://www.linkedin.com/feed/update/urn:li:ac...,Sponsored,"Boost Post Website Visits April 17, 2026 at 8:...",Ioanna Lykiardopoulou,04/16/2026,04/17/2026,04/24/2026,All followers,1381,NaN,NaN,0,0.0,1,0,0,0.0,0.004345,NaN
2,AI is high on every agenda. But for many organ...,https://www.linkedin.com/feed/update/urn:li:ac...,Total,NaN,Ioanna Lykiardopoulou,04/16/2026,NaN,NaN,All followers,1381,NaN,NaN,0,0.0,22,0,0,0.0,0.004345,NaN


In [27]:
posts_df.shape

(254, 20)